# aug21 — post-review bundle on Kaggle T4 x2 (B1 + B2 + C1.1 + C2 capture)

**Settings → Accelerator → GPU T4 x2, Internet → On.** Secrets: `HF_TOKEN`
(required), `WANDB_API_KEY` (recommended — B1's training log is a deliverable;
falls back to offline logging), `OPENROUTER_API_KEY` (optional, enables the C2
judge in-session).

Time budget ~10 h against Kaggle's 12 h cap: B1 training ~5.5 h (single T4,
original recipe) + serving/evals ~3 h + entropy/KL ~1 h. **If the session dies:
re-import and Run All — cell 3 resumes B1 from the Hub checkpoint repo, and the
later cells simply recompute.** Prerequisite: branch `analysis/aug21` pushed.


In [ ]:
# Cell 1 — setup
import os, glob, json, subprocess, time
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
try: os.environ['WANDB_API_KEY'] = S.get_secret('WANDB_API_KEY')
except Exception: os.environ['WANDB_MODE'] = 'offline'; print('no WANDB secret -> offline logging')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!git checkout analysis/aug21
assert os.path.exists('results-analysis/aug21/taskB1_grpo-7b-noscale.yaml'), 'push analysis/aug21 first'
!pip install -q -e . openai vllm peft trl bitsandbytes accelerate
!pip show trl vllm | grep -E "^(Name|Version)" | tee results-analysis/aug21/session-versions.txt
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits
# NO CLI login: V10 died here -- `huggingface-cli` (deprecated shim in hf_hub>=1.x)
# blocked 12h on an interactive 'update now? [Y/n]' prompt. HF_TOKEN in the env is
# read by huggingface_hub for every download/upload; no login step is needed.
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='artifacts/sft-7b',
                  token=os.environ['HF_TOKEN'])
print('SFT warm-start adapter in place (required by B1 init_adapter and entropy_kl)')

In [ ]:
# Cell 2 — ensure the scale_rewards plumbing exists AND is verifiable (R3a)
# Two layers: (a) conditional kwarg + signature guard, (b) READ-BACK off the
# constructed GRPOConfig object — the defect being guarded against is a value
# accepted and then discarded downstream, so only (b) actually proves the run
# is the ablation it claims to be.
p = 'src/connections_rl/train/grpo.py'
src = open(p).read()
changed = False
if 'scale_rewards' not in src:
    print('WARNING: applying the taskB1 patch IN-SESSION. Commit it to the branch too,')
    print('so the pushed training code matches what actually ran.')
    guard = (
        "    if \"scale_rewards\" in cfg:\n"
        "        import inspect\n"
        "        if \"scale_rewards\" not in inspect.signature(GRPOConfig.__init__).parameters:\n"
        "            raise RuntimeError(\n"
        "                \"config sets scale_rewards but this TRL version's GRPOConfig \"\n"
        "                \"does not accept it -- upgrade TRL; refusing to run an ablation \"\n"
        "                \"that would silently replicate the original.\"\n"
        "            )\n\n"
    )
    anchor_cfg = '    grpo_config = GRPOConfig('
    assert anchor_cfg in src, 'grpo.py layout changed; patch by hand'
    src = src.replace(anchor_cfg, guard + anchor_cfg, 1)
    anchor_kw = 'beta=cfg.get("kl_beta", 0.04),  # KL penalty to the reference policy'
    assert anchor_kw in src, 'kwargs anchor missing; patch by hand'
    src = src.replace(anchor_kw, anchor_kw +
        '\n                **({"scale_rewards": cfg["scale_rewards"]} if "scale_rewards" in cfg else {}),', 1)
    changed = True
if 'resolved on GRPOConfig' not in src:
    readback = (
        '    if "scale_rewards" in cfg:\n'
        '        import trl\n'
        '        _requested = cfg["scale_rewards"]\n'
        '        _resolved = getattr(grpo_config, "scale_rewards", "<attribute missing>")\n'
        '        print(f"[B1 scale_rewards] trl={trl.__version__}  "\n'
        '              f"requested={_requested!r}  resolved on GRPOConfig={_resolved!r}")\n'
        '        _equivalent = {_requested, {True: "group", False: "none"}.get(_requested, _requested)}\n'
        '        if _resolved not in _equivalent:\n'
        '            raise RuntimeError(\n'
        '                f"GRPOConfig resolved scale_rewards to {_resolved!r}, not the "\n'
        '                f"requested {_requested!r} or its documented normalization "\n'
        '                f"(trl {trl.__version__}). Refusing to train."\n'
        '            )\n\n'
    )
    anchor_after = '    # Cross-session resume: pull the latest checkpoint from the Hub before'
    assert anchor_after in src, 'post-construction anchor missing; patch by hand'
    src = src.replace(anchor_after, readback + anchor_after, 1)
    changed = True
if changed:
    open(p, 'w').write(src)
    print('patched (conditional kwarg + signature guard + read-back verification).')
else:
    print('grpo.py already carries kwarg + read-back (committed on the branch) -- good')
import importlib, connections_rl
print('sanity: module imports OK')


In [ ]:
# Cell 2b — R3a dry check, version-aware. Run BEFORE any GPU hours.
# The handoff's literal assert (`is False`) false-fails on trl >= the bool->str
# migration: GRPOConfig.__post_init__ maps {True: "group", False: "none"}
# (verified verbatim in trl v1.10.0 grpo_config.py). "none" IS the flag
# sticking -- no scaling applied. Anything else (e.g. "group") means it did not.
import inspect, trl
from trl import GRPOConfig
p = inspect.signature(GRPOConfig.__init__).parameters
print("trl:", trl.__version__, "| accepts scale_rewards:", "scale_rewards" in p)
c = GRPOConfig(output_dir="/tmp/_probe", scale_rewards=False)
default = GRPOConfig(output_dir="/tmp/_probe2").scale_rewards
print("requested False -> resolved:", repr(c.scale_rewards), "| default:", repr(default))
assert c.scale_rewards in (False, "none"), "FLAG DOES NOT STICK — do not launch"
print("OK — safe to launch")
with open('results-analysis/aug21/session-trl-version.txt', 'w') as f:
    f.write(f"trl {trl.__version__}; scale_rewards=False resolves to {c.scale_rewards!r}; "
            f"default is {default!r} (the paper's inherited-default claim, version-pinned)\n")


In [ ]:
# Cell 3 — B1 training (~5.5 h, single T4, original recipe; resumes from the Hub ckpt repo)
import os
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
r = subprocess.run(['python', '-m', 'connections_rl.train.grpo',
                    '--config', 'results-analysis/aug21/taskB1_grpo-7b-noscale.yaml'], env=env)
print('training exit code:', r.returncode)
ckpts = sorted(glob.glob('artifacts/grpo-7b-noscale/checkpoint-*'),
               key=lambda x: int(x.rsplit('-', 1)[1]))
print('local checkpoints:', [c.rsplit('-', 1)[1] for c in ckpts])
if r.returncode != 0 and not ckpts:
    raise SystemExit('B1 produced nothing -- one retry maximum, then report the death and stop.')

In [ ]:
# Cell 4 — serve EVERYTHING in one vLLM session (T4 x2, session-A recipe)
import subprocess, time, urllib.request, glob, os
ncs = sorted(glob.glob('artifacts/grpo-7b-noscale/checkpoint-*'), key=lambda x: int(x.rsplit('-', 1)[1]))
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-ckpt', local_dir='adapters/grpo-7b-ckpt', token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b', local_dir='adapters/grpo-7b', token=os.environ['HF_TOKEN'])
assert os.path.isdir('adapters/grpo-7b-ckpt/checkpoint-50') and os.path.isdir('adapters/grpo-7b-ckpt/checkpoint-100')

mods = ['connections-rl-grpo-7b-ckpt50=adapters/grpo-7b-ckpt/checkpoint-50',
        'connections-rl-grpo-7b-ckpt100=adapters/grpo-7b-ckpt/checkpoint-100',
        'connections-rl-grpo-7b=adapters/grpo-7b',
        'connections-rl-grpo-7b-noscale=artifacts/grpo-7b-noscale']
for c in ncs:
    step = c.rsplit('-', 1)[1]
    mods.append(f'noscale-ckpt-{step}={c}')
proc = subprocess.Popen(
    'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
    '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
    shell=True, stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(150):
    try: urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); break
    except Exception: time.sleep(10)
else: raise RuntimeError('vLLM failed -- see /kaggle/working/vllm.log')

In [ ]:
# Cell 5 — B1 checkpoint curve (val) -> auto-pick peak -> all evals + C2 capture + stats
arms_arg = 'connections-rl-sft-7b:0,' if False else ''  # SFT point optional; ckpts carry the story
arms_arg += ','.join(f"noscale-ckpt-{c.rsplit('-',1)[1]}:{c.rsplit('-',1)[1]}" for c in ncs)
!python -m connections_rl.eval.checkpoint_curve --arms {arms_arg} \
    --puzzles data/splits/puzzles_val.json --out results-analysis/aug21/ckpt-curve-7b-noscale
pts = json.load(open('results-analysis/aug21/ckpt-curve-7b-noscale.json'))
peak = max(pts, key=lambda p: p['semantic_groups_correct'])
PEAK = peak['step']
print(f"B1 val-semantic peak: step {PEAK} (semantic {peak['semantic_groups_correct']:.4f})")
print('DELIVERABLE ANSWER (hump): does semantic rise then fall?',
      [round(p['semantic_groups_correct'], 4) for p in pts])

cfg = open('results-analysis/aug21/taskB2_eval_test.yaml').read()
cfg = cfg.replace('model: connections-rl-grpo-7b-noscale-peak   # ckpt NOSCALE_PEAK_STEP',
                  f'model: noscale-ckpt-{PEAK}')
open('results-analysis/aug21/taskB2_eval_test.resolved.yaml', 'w').write(cfg)

!python results-analysis/aug21/c1_shift_analysis.py prepare
!python -m connections_rl.eval.run --config results-analysis/aug21/taskB2_eval_test.resolved.yaml
!python -m connections_rl.eval.run --config results-analysis/aug21/c11_eval_trainslice.yaml
!python results-analysis/aug21/c2_capture_generations.py
!python results-analysis/aug21/taskB2_paired.py
!python results-analysis/aug21/c1_shift_analysis.py analyze
# R-handoff: map session outputs to the exact data/ paths (with the R6
# 26/77/4-of-648 gate) and print the two explicit R3b answers.
!python results-analysis/aug21/r_map_deliverables.py
pg = json.load(open('results-analysis/aug21/evalB-session/paired_groups.json'))['comparisons']
print('R3b ANSWER 1 (hump): val semantic by step:',
      [(pt['step'], round(pt['semantic_groups_correct'], 4)) for pt in pts])
print('R3b ANSWER 2 (below base): noscale-final minus base paired diff (0-4, seed 0):',
      pg.get('noscale-final_minus_base'))


In [ ]:
# Cell 6 — kill the server, then the B1 entropy/KL series (PEFT path, ~1 h)
proc.terminate(); time.sleep(20)
ck_args = ' '.join(f"ckpt-{c.rsplit('-',1)[1]}={c}" for c in ncs)
!python -m connections_rl.eval.entropy_kl \
    --model Qwen/Qwen2.5-7B-Instruct --load-in-4bit --sft-adapter artifacts/sft-7b \
    --checkpoints base=base sft=artifacts/sft-7b {ck_args} \
    --puzzles data/splits/puzzles_val.json --n 100 --temperature 0.9 \
    --out results-analysis/aug21/entropy-kl-7b-noscale
# (artifacts/sft-7b was downloaded in cell 1)

In [ ]:
# Cell 7 — optional in-session extras: C2 judge + B3 W&B export (skip silently if no secret)
try:
    os.environ['OPENROUTER_API_KEY'] = S.get_secret('OPENROUTER_API_KEY')
    !python results-analysis/aug21/c2_judge_openrouter.py
except Exception as e:
    print('C2 judging skipped here (run locally with your key):', e)
try:
    S.get_secret('WANDB_API_KEY')
    !pip install -q wandb && python results-analysis/aug21/b3_wandb_export.py
except Exception as e:
    print('B3 export skipped here (run locally):', e)

In [ ]:
# Cell 8 — persist: zip + Hub dataset copy
!zip -qr /kaggle/working/aug21-outputs.zip results-analysis/aug21 data -x 'data/splits/*'
from huggingface_hub import HfApi
from IPython.display import FileLink, display
HfApi().upload_folder(folder_path='results-analysis/aug21',
                      repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset',
                      path_in_repo='aug21')
print(f'pushed -> huggingface.co/datasets/{HF_USER}/connections-rl-results/tree/main/aug21')
display(FileLink('/kaggle/working/aug21-outputs.zip'))
HfApi().upload_folder(folder_path='data', repo_id=f'{HF_USER}/connections-rl-results',
                      repo_type='dataset', path_in_repo='aug21/data',
                      ignore_patterns=['splits/*'])
print(f'data/ mirrored -> huggingface.co/datasets/{HF_USER}/connections-rl-results/tree/main/aug21/data')


**Afterwards:** tell the agent — it pulls `aug21/` from the Hub, re-verifies
every number from the records, and writes the report (B1 hump + below-base
answer, B2 plateau-vs-fall, C1 shift trio, C2 judge ranking).

**If the 12 h cap killed the session mid-training:** re-import, Run All. Cell 3
resumes from `connections-rl-grpo-7b-noscale-ckpt` on the Hub; everything after
recomputes. One retry maximum on a genuine training death — A–E and Task D are
already banked and worth more than this run.